# Module: Uninformed Search

Uninformed search can generate successors and recognise the goal, nothing more. The four
algorithms below differ only in which node they take out of the frontier next.

| # | Algorithm | Frontier |
|---|---|---|
| 1 | Depth-First Search (DFS) | stack (LIFO) |
| 2 | Breadth-First Search (BFS) | queue (FIFO) |
| 3 | Iterative Deepening Search (IDS) | stack, re-run with a growing depth limit |
| 4 | Uniform-Cost Search (UCS) | priority queue on path cost g(n) |

Graphs are adjacency dicts: `graph[u][v]` is the cost of the edge between u and v. Each function
returns `(path, cost, nodes_expanded)`, where a node counts as expanded when we generate its
successors. Neighbours are visited in alphabetical order, so every run gives the same result.

<h2><b>1. Depth-First Search (DFS)</b></h2>

![](data\dfs.gif)

GIF source: https://takeuforward.org/data-structure/depth-first-search-dfs/

DFS follows one branch as deep as it can. At a dead end it backs up to the latest node that still
has an untried neighbour.

The frontier is a stack (LIFO): the path pushed last comes out first. This version keeps an
`explored` set so cycles cannot trap it. DFS ignores edge costs and returns the first path it
reaches.

**Step-by-step mechanism of DFS:**

- Initialise the stack with the path `[start]` and an empty `explored` set.
- Loop while the stack is not empty:
  - Pop the path on top of the stack; let `node` be its last node.
  - If `node` is the goal, return the path (**goal found**).
  - If `node` is already in `explored`, skip it (it was reached earlier by another path).
  - Add `node` to `explored` and count it as expanded.
  - Push `path + [neighbor]` for every unexplored neighbour, in reverse alphabetical order, so the
    alphabetically first neighbour ends up on top and is explored next.
- If the stack becomes empty, every reachable node has been explored: return **failure**.

**Properties of DFS**

| Complete? | Optimal? | Time | Space |
|---|---|---|---|
| Yes on finite graphs; no on infinite ones | No | O(b<sup>m</sup>) | O(b·m) |

- **Complete?** The explored set makes it finish on a finite graph; in an infinite space it can dive forever.
- **Optimal?** It returns the first path found, whatever that path costs.
- **Time:** it may explore the whole tree down to depth m, and m can be much larger than d.
- **Space:** the tree-search version stores one path plus siblings, O(b·m); the `explored` set in this version can grow to the size of the graph.

**Case study.** Robo, a museum security robot, is at the entrance (room **A**) when an alarm goes
off at the emergency exit (room **H**). Robo has no map, so it takes the first corridor on its list
and backs up only at dead ends. Edge costs are corridor lengths in tens of metres.

In [1]:
museum = {
    'A': {'B': 2, 'C': 4},
    'B': {'A': 2, 'D': 3, 'E': 6},
    'C': {'A': 4, 'F': 2},
    'D': {'B': 3, 'G': 5},
    'E': {'B': 6, 'H': 3},
    'F': {'C': 2, 'H': 4},
    'G': {'D': 5},
    'H': {'E': 3, 'F': 4},
}

In [2]:
def dfs(graph, start, goal, verbose=False):
    """Depth-first graph search. Returns (path, cost, nodes_expanded)."""
    stack = [[start]]                 # LIFO frontier of paths
    explored = set()
    nodes_expanded = 0

    while stack:
        path = stack.pop()            # most recently pushed path
        node = path[-1]
        if node == goal:
            cost = sum(graph[u][v] for u, v in zip(path, path[1:]))
            return path, cost, nodes_expanded
        if node in explored:
            continue
        explored.add(node)
        nodes_expanded += 1
        if verbose:
            print(f"expand {node:<3} stack before push: {[p[-1] for p in stack]}")
        # reverse order so the alphabetically first neighbour is popped first
        for neighbor in sorted(graph[node], reverse=True):
            if neighbor not in explored:
                stack.append(path + [neighbor])

    return None, float('inf'), nodes_expanded

In [3]:
path, cost, expanded = dfs(museum, 'A', 'H', verbose=True)
print()
print("Path found     :", " -> ".join(path))
print("Path cost      :", cost)
print("Nodes expanded :", expanded)

expand A   stack before push: []
expand B   stack before push: ['C']
expand D   stack before push: ['C', 'E']
expand G   stack before push: ['C', 'E']
expand E   stack before push: ['C']

Path found     : A -> B -> E -> H
Path cost      : 11
Nodes expanded : 5


<h2><b>2. Breadth-First Search (BFS)</b></h2>

![](data\bfs.gif)

GIF source: https://takeuforward.org/graph/breadth-first-search-bfs-level-order-traversal

BFS works level by level: every node one edge from the start, then every node two edges away, and
so on. The first path it finds to the goal has the fewest edges.

The frontier is a FIFO queue (`collections.deque`). Swapping DFS's stack for a queue is the only
change, and the search goes wide instead of deep. BFS no longer gets lost in a long branch, but it
has to hold a whole level in memory. Shallow paths always leave the queue first, so BFS can test
for the goal as soon as it generates a node.

**Step-by-step mechanism of BFS:**

- If the start is the goal, return `[start]` immediately.
- Initialise the queue with the path `[start]` and a `reached` set containing `start`.
- Loop while the queue is not empty:
  - Dequeue the path at the front; let `node` be its last node and count it as expanded.
  - For every neighbour (alphabetical order) not yet in `reached`:
    - Build `new_path = path + [neighbor]`.
    - If the neighbour is the goal, return `new_path` (**goal found**, early goal test).
    - Otherwise add the neighbour to `reached` and enqueue `new_path` at the back.
- If the queue becomes empty, return **failure**.

**Properties of BFS**

| Complete? | Optimal? | Time | Space |
|---|---|---|---|
| Yes (finite b) | Only when all step costs are equal | O(b<sup>d</sup>) | O(b<sup>d</sup>) |

- **Complete?** It finishes each level before the next, so it reaches a goal at depth d.
- **Optimal?** Fewest edges means lowest cost only when every edge costs the same.
- **Time:** with the early goal test it generates at most b + b<sup>2</sup> + … + b<sup>d</sup> nodes.
- **Space:** it keeps every generated node, so memory usually runs out before time does.

**Case study.** Rina lands at the **Airport** and wants to reach the **Zoo** with as few stops as
possible. Edge costs are travel minutes. BFS ignores them, but we print the total to see what the
choice costs her.

In [4]:
metro = {
    'Airport':    {'Central': 20, 'Harbor': 6},
    'Central':    {'Airport': 20, 'University': 7, 'Zoo': 18},
    'Harbor':     {'Airport': 6, 'Museum': 5},
    'Museum':     {'Harbor': 5, 'Park': 4},
    'Park':       {'Museum': 4, 'Zoo': 5},
    'University': {'Central': 7},
    'Zoo':        {'Central': 18, 'Park': 5},
}

In [5]:
from collections import deque

def bfs(graph, start, goal, verbose=False):
    """Breadth-first graph search with early goal test. Returns (path, cost, nodes_expanded)."""
    if start == goal:
        return [start], 0, 0
    frontier = deque([[start]])       # FIFO frontier of paths
    reached = {start}
    nodes_expanded = 0

    while frontier:
        path = frontier.popleft()     # oldest path first
        node = path[-1]
        nodes_expanded += 1
        if verbose:
            print(f"expand {node:<11} queue: {[p[-1] for p in frontier]}")
        for neighbor in sorted(graph[node]):
            if neighbor not in reached:
                new_path = path + [neighbor]
                if neighbor == goal:              # early goal test
                    cost = sum(graph[u][v] for u, v in zip(new_path, new_path[1:]))
                    return new_path, cost, nodes_expanded
                reached.add(neighbor)
                frontier.append(new_path)

    return None, float('inf'), nodes_expanded

In [6]:
path, cost, expanded = bfs(metro, 'Airport', 'Zoo', verbose=True)
print()
print("Path found     :", " -> ".join(path))
print("Stations passed:", len(path) - 1)
print("Travel minutes :", cost)
print("Nodes expanded :", expanded)

expand Airport     queue: []
expand Central     queue: ['Harbor']

Path found     : Airport -> Central -> Zoo
Stations passed: 2
Travel minutes : 38
Nodes expanded : 2


Rina gets there in 2 stops but spends 38 minutes on the train. Airport → Harbor → Museum → Park →
Zoo takes 4 stops and 20 minutes.

<h2><b>3. Iterative Deepening Search (IDS)</b></h2>

![](data\ids.gif)

GIF source: <TODO>

IDS or IDDFS (Iterative Deepening Depth-First Search) runs a depth-limited DFS with limit 0, then 1, then 2, until it finds the goal. It returns the
same shallowest goal as BFS.

It still uses a stack (here Python's call stack, through recursion) and only rejects nodes already
on the current path. Compared with BFS, it stores one path instead of a whole level. Each round
re-expands the shallow nodes, but the deepest level holds most of the nodes, so the extra work is
a constant factor.

**Step-by-step mechanism of IDS:**

- Set the depth limit to 0.
- Run a depth-limited search (DLS) from the start:
  - If the current node is the goal, return the path.
  - If the remaining limit is 0, return `cutoff` (the goal might still be deeper).
  - Otherwise count the node as expanded and call DLS on each neighbour (alphabetical order) that
    is not already on the current path, with the limit reduced by 1.
  - If a child returns a path, pass it upward; remember whether any child returned `cutoff`.
- If DLS returned a path, return it (**goal found**).
- If DLS returned `cutoff`, increase the limit by 1 and run DLS again from scratch.
- If DLS returned plain failure (no cutoff anywhere), the whole graph was searched: return
  **failure**.

**Properties of IDS**

| Complete? | Optimal? | Time | Space |
|---|---|---|---|
| Yes (finite b) | Only when all step costs are equal | O(b<sup>d</sup>) | O(b·d) |

- **Complete?** The limit keeps growing, so round d reaches a goal at depth d.
- **Optimal?** Same condition as BFS: equal step costs.
- **Time:** nodes at depth k are regenerated d − k + 1 times, which still sums to O(b<sup>d</sup>).
- **Space:** each round is a DFS capped at depth d.

**Case study.** Kiko, a rescue drone, enters a cave to find a trapped hiker (**Survivor**) starting
from the **Entrance**. The battery allows few hops and the drone cannot store a full map, so IDS
suits it. Edge costs are tunnel lengths in tens of metres.

In [7]:
cave = {
    'Entrance': {'North': 2, 'South': 3},
    'North':    {'Entrance': 2, 'Pool': 4, 'Shaft': 2},
    'South':    {'Entrance': 3, 'Crystal': 3, 'Lake': 5},
    'Pool':     {'North': 4},
    'Shaft':    {'North': 2, 'Deep': 6},
    'Crystal':  {'South': 3, 'Survivor': 4},
    'Lake':     {'South': 5, 'Survivor': 2},
    'Deep':     {'Shaft': 6, 'Survivor': 3},
    'Survivor': {'Crystal': 4, 'Lake': 2, 'Deep': 3},
}

In [8]:
def ids(graph, start, goal, max_depth=50, verbose=False):
    """Iterative deepening search. Returns (path, cost, nodes_expanded) summed over all iterations."""
    nodes_expanded = 0

    def depth_limited(path, limit):
        nonlocal nodes_expanded
        node = path[-1]
        if node == goal:
            return path
        if limit == 0:
            return 'cutoff'
        nodes_expanded += 1
        cutoff_occurred = False
        for neighbor in sorted(graph[node]):
            if neighbor in path:              # avoid cycles on the current path
                continue
            result = depth_limited(path + [neighbor], limit - 1)
            if result == 'cutoff':
                cutoff_occurred = True
            elif result is not None:
                return result
        return 'cutoff' if cutoff_occurred else None

    for limit in range(max_depth + 1):
        result = depth_limited([start], limit)
        if verbose:
            print(f"limit = {limit}: total nodes expanded so far = {nodes_expanded}")
        if result is None:
            return None, float('inf'), nodes_expanded
        if result != 'cutoff':
            cost = sum(graph[u][v] for u, v in zip(result, result[1:]))
            return result, cost, nodes_expanded
    return None, float('inf'), nodes_expanded

In [9]:
path, cost, expanded = ids(cave, 'Entrance', 'Survivor', verbose=True)
print()
print("Path found     :", " -> ".join(path))
print("Hops           :", len(path) - 1)
print("Path cost      :", cost)
print("Nodes expanded :", expanded)

print()
dfs_path, dfs_cost, dfs_expanded = dfs(cave, 'Entrance', 'Survivor')
print("For contrast, plain DFS finds:", " -> ".join(dfs_path), f"({len(dfs_path) - 1} hops)")

limit = 0: total nodes expanded so far = 0
limit = 1: total nodes expanded so far = 1
limit = 2: total nodes expanded so far = 4
limit = 3: total nodes expanded so far = 10

Path found     : Entrance -> South -> Crystal -> Survivor
Hops           : 3
Path cost      : 10
Nodes expanded : 10

For contrast, plain DFS finds: Entrance -> North -> Shaft -> Deep -> Survivor (4 hops)


<h2><b>4. Uniform-Cost Search (UCS)</b></h2>

![](data\ucs.gif)

GIF source: https://github.com/aghdamamir/UCS-and-A-star-search

UCS expands the node with the lowest path cost g(n). If every edge costs the same, it behaves like
BFS.

The frontier is a priority queue (`heapq`) keyed on g(n). A node can sit in the queue more than
once; the code skips entries made stale by a cheaper path. Unlike BFS and IDS, UCS reads edge
costs and returns the cheapest path. It tests for the goal when it pops a node, because a cheaper
path to the goal may still be waiting in the queue.

**Step-by-step mechanism of UCS:**

- Push `(0, [start])` onto the priority queue and set `best_cost[start] = 0`.
- Loop while the queue is not empty:
  - Pop the entry with the smallest cost g; let `node` be the last node of its path.
  - If g is larger than `best_cost[node]`, this entry is stale: skip it.
  - If `node` is the goal, return the path and g (**goal found**, late goal test).
  - Count `node` as expanded. For each neighbour, compute `new_cost = g + step_cost`.
  - If `new_cost` beats the best known cost for that neighbour, record it and push
    `(new_cost, path + [neighbor])`.
- If the queue becomes empty, return **failure**.

**Properties of UCS**

| Complete? | Optimal? | Time | Space |
|---|---|---|---|
| Yes (every step cost ≥ ε > 0) | Yes | O(b<sup>1 + ⌊C*/ε⌋</sup>) | O(b<sup>1 + ⌊C*/ε⌋</sup>) |

- **Complete?** With every step ≥ ε, an infinite path has unbounded cost, so UCS cannot get stuck on one.
- **Optimal?** It pops nodes in order of g, so the first goal it pops is the cheapest.
- **Time:** many cheap steps make ⌊C*/ε⌋ large, so UCS can be far slower than BFS.
- **Space:** the frontier follows the same bound.

**Case study.** Budi, a delivery rider, carries a parcel from the **Warehouse** to a **Customer**.
He pays for his own fuel, so he wants the cheapest route even if it passes more junctions. Edge
costs are fuel costs.

In [10]:
roads = {
    'Warehouse': {'Bakery': 2, 'Clinic': 6, 'Dock': 9},
    'Bakery':    {'Warehouse': 2, 'Clinic': 3, 'Farm': 7},
    'Clinic':    {'Warehouse': 6, 'Bakery': 3, 'Gym': 2},
    'Dock':      {'Warehouse': 9, 'Customer': 3},
    'Farm':      {'Bakery': 7, 'Customer': 4},
    'Gym':       {'Clinic': 2, 'Customer': 3},
    'Customer':  {'Dock': 3, 'Farm': 4, 'Gym': 3},
}

In [11]:
import heapq

def ucs(graph, start, goal, verbose=False):
    """Uniform-cost search. Returns (path, cost, nodes_expanded)."""
    frontier = [(0, [start])]         # priority queue ordered by g(n)
    best_cost = {start: 0}
    nodes_expanded = 0

    while frontier:
        cost, path = heapq.heappop(frontier)
        node = path[-1]
        if cost > best_cost[node]:    # stale entry, a cheaper path was found later
            continue
        if node == goal:              # late goal test guarantees optimality
            return path, cost, nodes_expanded
        nodes_expanded += 1
        if verbose:
            print(f"expand {node:<10} g = {cost:>2}")
        for neighbor, step in sorted(graph[node].items()):
            new_cost = cost + step
            if new_cost < best_cost.get(neighbor, float('inf')):
                best_cost[neighbor] = new_cost
                heapq.heappush(frontier, (new_cost, path + [neighbor]))

    return None, float('inf'), nodes_expanded

In [12]:
path, cost, expanded = ucs(roads, 'Warehouse', 'Customer', verbose=True)
print()
print("Path found     :", " -> ".join(path))
print("Fuel cost      :", cost)
print("Nodes expanded :", expanded)

print()
bfs_path, bfs_cost, _ = bfs(roads, 'Warehouse', 'Customer')
print("For contrast, BFS finds:", " -> ".join(bfs_path), f"(fuel cost {bfs_cost})")

expand Warehouse  g =  0
expand Bakery     g =  2
expand Clinic     g =  5
expand Gym        g =  7
expand Farm       g =  9
expand Dock       g =  9

Path found     : Warehouse -> Bakery -> Clinic -> Gym -> Customer
Fuel cost      : 10
Nodes expanded : 6

For contrast, BFS finds: Warehouse -> Dock -> Customer (fuel cost 12)


## Comparison

All four algorithms on one graph: a courier delivering a package from city **A** to city **J**.

In [13]:
courier_map = {
    'A': {'B': 4, 'C': 3, 'D': 7},
    'B': {'A': 4, 'E': 5},
    'C': {'A': 3, 'F': 2, 'G': 6},
    'D': {'A': 7, 'J': 14},
    'E': {'B': 5, 'H': 4},
    'F': {'C': 2, 'G': 2, 'I': 5},
    'G': {'C': 6, 'F': 2, 'J': 6},
    'H': {'E': 4, 'J': 3},
    'I': {'F': 5, 'J': 4},
    'J': {'D': 14, 'G': 6, 'H': 3, 'I': 4},
}

In [14]:
def print_table(rows):
    header = f"{'algorithm':<22} | {'path':<40} | {'cost':>5} | {'nodes expanded':>14}"
    print(header)
    print("-" * len(header))
    for name, path, cost, expanded in rows:
        path_str = " -> ".join(path) if path else "no solution"
        print(f"{name:<22} | {path_str:<40} | {cost:>5} | {expanded:>14}")

rows = []
for name, algorithm in [('DFS', dfs), ('BFS', bfs), ('IDS', ids), ('UCS', ucs)]:
    path, cost, expanded = algorithm(courier_map, 'A', 'J')
    rows.append((name, path, cost, expanded))
print_table(rows)

algorithm              | path                                     |  cost | nodes expanded
------------------------------------------------------------------------------------------
DFS                    | A -> B -> E -> H -> J                    |    16 |              4
BFS                    | A -> D -> J                              |    21 |              4
IDS                    | A -> D -> J                              |    21 |              5
UCS                    | A -> C -> F -> G -> J                    |    13 |              9


DFS, BFS and IDS finish after 4 or 5 expansions but ignore edge costs. DFS returns whatever path
alphabetical order reaches first (cost 16). BFS and IDS both return A → D → J, the path with the
fewest edges and here the most expensive one (21); IDS needs one extra expansion but far less
memory. UCS expands 9 nodes and is the only one to return the cheapest route,
A → C → F → G → J (13).